In [2]:
!pip install selenium webdriver-manager

In [14]:
!apt-get update -qq
!apt-get install -y -qq chromium
!pip install -q -U selenium

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Preconfiguring packages ...
Selecting previously unselected package apparmor.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../apparmor_4.0.1really4.0.1-0ubuntu0.24.04.8_amd64.deb ...
Unpacking apparmor (4.0.1really4.0.1-0ubuntu0.24.04.8) ...
Selecting previously unselected package squashfs-tools.
Preparing to unpack .../squashfs-tools_1%3a4.6.1-1build1_amd64.deb ...
Unpacking squashfs-tools (1:4.6.1-1build1) ...
Selecting previously unselected package udev.
Preparing to unpack .../udev_255.4-1ubuntu8.17_amd64.deb ...
Unpacking udev (255.4-1ubuntu8.17) ...
Selecting previously unselected package libfuse3-3:amd64.
Preparing to unpack .../libfuse3-3_3.14.0-5build1_amd64.deb ...
Unpacking libfuse3-3:amd64 (3.14.0-5build1) ...
Selecting previously unselecte

In [15]:
!which chromium
!chromium --version

/bin/bash: line 1: chromium: command not found


In [16]:
%pip install -q google-colab-selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 19.8 MB/s eta 0:00:00


In [3]:
import os

folders = ["src", "configs", "data", "docs"]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

with open("src/__init__.py", "w", encoding="utf-8") as f:
    pass

print("Project folders created successfully!")

Project folders created successfully!


In [17]:
%%writefile src/driver_manager.py
import google_colab_selenium as gs


class DriverManager:
    """Manage Chrome WebDriver in Google Colab."""

    @staticmethod
    def create_driver():
        driver = gs.Chrome()
        driver.set_page_load_timeout(30)
        return driver

Overwriting src/driver_manager.py


In [18]:
from src.driver_manager import DriverManager

driver = DriverManager.create_driver()

try:
    driver.get("https://books.toscrape.com/")
    print("Title:", driver.title)
    print("URL:", driver.current_url)
finally:
    driver.quit()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Title: All products | Books to Scrape - Sandbox
URL: https://books.toscrape.com/


In [5]:
%%writefile src/config_parser.py
import json
from pathlib import Path


class ConfigParser:
    """Load and validate scraper configuration."""

    @staticmethod
    def load_config(config_path):
        path = Path(config_path)

        if not path.exists():
            raise FileNotFoundError(
                f"Configuration file not found: {config_path}"
            )

        with path.open("r", encoding="utf-8") as file:
            config = json.load(file)

        required_keys = [
            "start_url",
            "max_pages",
            "delay",
            "item_container_selector",
            "item_data_selectors",
        ]

        for key in required_keys:
            if key not in config:
                raise ValueError(f"Missing configuration key: {key}")

        if not isinstance(config["max_pages"], int) or config["max_pages"] < 1:
            raise ValueError("max_pages must be a positive integer")

        if not isinstance(config["delay"], (int, float)) or config["delay"] < 0:
            raise ValueError("delay must be a non-negative number")

        if not isinstance(config["item_data_selectors"], dict):
            raise ValueError("item_data_selectors must be a dictionary")

        for field, selector in config["item_data_selectors"].items():
            if selector is not None and not isinstance(selector, str):
                raise ValueError(
                    f"Selector for '{field}' must be a string or null"
                )

        return config

Writing src/config_parser.py


In [6]:
%%writefile src/data_models.py
from dataclasses import dataclass, asdict
from typing import Optional


@dataclass
class ScrapedItem:
    name: Optional[str] = None
    price: Optional[str] = None
    description: Optional[str] = None
    url: Optional[str] = None
    image: Optional[str] = None

    def to_dict(self):
        return asdict(self)

Writing src/data_models.py


In [7]:
%%writefile src/utils.py
import json
import os
from datetime import datetime


def save_to_json(data, output_path):
    """Save scraped data as a JSON file."""
    directory = os.path.dirname(output_path)

    if directory:
        os.makedirs(directory, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=4)

    print(f"Saved {len(data)} items to: {output_path}")


def log_message(message):
    """Print a timestamped log message."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{timestamp}] {message}")

Writing src/utils.py


In [8]:
%%writefile configs/example_site_config.json
{
    "start_url": "https://books.toscrape.com/catalogue/category/books_1/index.html",
    "max_pages": 3,
    "delay": 2,
    "item_container_selector": "article.product_pod",
    "pagination_selector": "li.next a",
    "item_data_selectors": {
        "name": "h3 a",
        "price": "p.price_color",
        "description": null,
        "url": "h3 a",
        "image": "div.image_container a img"
    }
}

Writing configs/example_site_config.json


In [9]:
%%writefile src/scraper_agent.py
import time
from urllib.parse import urljoin

from selenium.webdriver.common.by import By
from selenium.common.exceptions import (
    NoSuchElementException,
    TimeoutException,
)

from .driver_manager import DriverManager
from .data_models import ScrapedItem
from .utils import log_message


class ScraperAgent:
    """Scrape items from a website using configuration settings."""

    def __init__(self, config):
        self.config = config
        self.driver = None
        self.scraped_items = []

    def _get_text(self, container, selector):
        if not selector:
            return None

        try:
            element = container.find_element(By.CSS_SELECTOR, selector)
            value = element.text.strip()
            return value or None
        except NoSuchElementException:
            return None

    def _get_attribute(self, container, selector, attribute, base_url):
        if not selector:
            return None

        try:
            element = container.find_element(By.CSS_SELECTOR, selector)
            value = element.get_attribute(attribute)

            if not value:
                return None

            if attribute in ("href", "src"):
                return urljoin(base_url, value)

            return value.strip() if isinstance(value, str) else value

        except NoSuchElementException:
            return None

    def _scrape_current_page(self):
        current_url = self.driver.current_url
        selector = self.config["item_container_selector"]

        containers = self.driver.find_elements(By.CSS_SELECTOR, selector)
        log_message(f"Found {len(containers)} items on {current_url}")

        selectors = self.config["item_data_selectors"]

        for container in containers:
            item = ScrapedItem(
                name=(
                    self._get_attribute(
                        container, selectors.get("name"), "title", current_url
                    )
                    or self._get_text(
                        container, selectors.get("name")
                    )
                ),
                price=self._get_text(
                    container, selectors.get("price")
                ),
                description=self._get_text(
                    container, selectors.get("description")
                ),
                url=self._get_attribute(
                    container, selectors.get("url"), "href", current_url
                ),
                image=self._get_attribute(
                    container, selectors.get("image"), "src", current_url
                ),
            )

            self.scraped_items.append(item.to_dict())

    def run(self):
        max_pages = self.config["max_pages"]
        delay = self.config["delay"]
        next_selector = self.config.get("pagination_selector")
        current_url = self.config["start_url"]

        self.driver = DriverManager.create_driver()

        try:
            for page_number in range(1, max_pages + 1):
                log_message(
                    f"Opening page {page_number}/{max_pages}: {current_url}"
                )

                self.driver.get(current_url)
                time.sleep(delay)

                self._scrape_current_page()

                if page_number >= max_pages or not next_selector:
                    break

                next_links = self.driver.find_elements(
                    By.CSS_SELECTOR, next_selector
                )

                if not next_links:
                    log_message("No next page link found. Stopping.")
                    break

                next_href = next_links[0].get_attribute("href")

                if not next_href:
                    log_message("Next page URL is empty. Stopping.")
                    break

                next_url = urljoin(self.driver.current_url, next_href)

                if next_url == current_url:
                    log_message("Next page URL has not changed. Stopping.")
                    break

                current_url = next_url

        except TimeoutException:
            log_message("Page loading timed out. Ending the scraping process.")

        finally:
            self.driver.quit()

        log_message(
            f"Scraping completed. Total items: {len(self.scraped_items)}"
        )
        return self.scraped_items

Writing src/scraper_agent.py


In [10]:
%%writefile main.py
from src.config_parser import ConfigParser
from src.scraper_agent import ScraperAgent
from src.utils import save_to_json, log_message


def main():
    config_path = "configs/example_site_config.json"
    output_path = "data/scraped_products.json"

    log_message("Starting Web Scraper Agent")

    config = ConfigParser.load_config(config_path)
    agent = ScraperAgent(config)

    items = agent.run()
    save_to_json(items, output_path)

    log_message("Program finished")


if __name__ == "__main__":
    main()

Writing main.py


In [11]:
import os

for root, dirs, files in os.walk("."):
    dirs[:] = [
        d for d in dirs
        if d not in ("sample_data", ".config", "__pycache__")
    ]

    level = root.count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

    for filename in files:
        print(f"{indent}    {filename}")

./
    main.py
    configs/
        example_site_config.json
    data/
    src/
        config_parser.py
        __init__.py
        data_models.py
        driver_manager.py
        scraper_agent.py
        utils.py
    docs/


In [19]:
!python main.py

[2026-10-09 14:59:41] Starting Web Scraper Agent
<IPython.core.display.HTML object>
Hit:1 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Get:4 https://dl.google.com/linux/chrome-stable/deb stable InRelease [2,548 B]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Get:8 https://dl.google.com/linux/chrome-stable/deb stable/main amd64 Packages [1,407 B]
Hit:9 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Fetched 3,955 B in 1s (7,697 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.li

In [20]:
import json

with open("data/scraped_products.json", "r", encoding="utf-8") as file:
    products = json.load(file)

print("Total products:", len(products))
print("\nExample product:")

if products:
    print(json.dumps(products[0], ensure_ascii=False, indent=4))

Total products: 60

Example product:
{
    "name": "A Light in the Attic",
    "price": "£51.77",
    "description": null,
    "url": "https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html",
    "image": "https://books.toscrape.com/media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"
}


In [21]:
import pandas as pd
from IPython.display import display

df = pd.DataFrame(products)
display(df)

,name,price,description,url,image
0,A Light in the Attic,£51.77,None,https://books.toscrape.com/catalogue/a-light-i...,https://books.toscrape.com/media/cache/2c/da/2...
1,Tipping the Velvet,£53.74,None,https://books.toscrape.com/catalogue/tipping-t...,https://books.toscrape.com/media/cache/26/0c/2...
2,Soumission,£50.10,None,https://books.toscrape.com/catalogue/soumissio...,https://books.toscrape.com/media/cache/3e/ef/3...
3,Sharp Objects,£47.82,None,https://books.toscrape.com/catalogue/sharp-obj...,https://books.toscrape.com/media/cache/32/51/3...
4,Sapiens: A Brief History of Humankind,£54.23,None,https://books.toscrape.com/catalogue/sapiens-a...,https://books.toscrape.com/media/cache/be/a5/b...
5,The Requiem Red,£22.65,None,https://books.toscrape.com/catalogue/the-requi...,https://books.toscrape.com/media/cache/68/33/6...
6,The Dirty Little Secrets of Getting Your Dream...,£33.34,None,https://books.toscrape.com/catalogue/the-dirty...,https://books.toscrape.com/media/cache/92/27/9...
7,The Coming Woman: A Novel Based on the Life of...,£17.93,None,https://books.toscrape.com/catalogue/the-comin...,https://books.toscrape.com/media/cache/3d/54/3...
8,The Boys in the Boat: Nine Americans and Their...,£22.60,None,https://books.toscrape.com/catalogue/the-boys-...,https://books.toscrape.com/media/cache/66/88/6...
9,The Black Maria,£52.15,None,https://books.toscrape.com/catalogue/the-black...,https://books.toscrape.com/media/cache/58/46/5...


In [22]:
from google.colab import files

files.download("data/scraped_products.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>